# Create an AI Search Endpoint and Index for RAG

**Objective**: Create an AI Search (previously called Vector Search) endpoint and Delta Sync index from the `dev.kumaflow_gold.transcript_chunks` gold table to enable semantic search and RAG over Kumaflow's meeting transcript chunks.

**AI Search** is Databricks' managed vector database. It stores an embedding (a numeric vector) for each chunk and returns those whose meaning is closest to a query, even when they share no keywords. It is the retrieval layer for RAG and consists of 2 components:
- **Endpoint**: the serverless compute that hosts one or more indexes and answers queries
- **Index**: the searchable vectors for one source table, stored in Unity Catalog and governed like any other table.

A **Delta Sync Index** stays in sync with a source Delta table automatically. It reads the table's Change Data Feed, so rows you insert, update or delete in `transcript_chunks` are reflected in the index without writing any sync code. The **Direct Vector Access Index** is the alternative option that you manage yourself.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType,
    EndpointStatusState,
    VectorIndexType,
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    PipelineType,
)
import time

w = WorkspaceClient()

endpoint_name = "transcript_embeddings_demo"
index_name = "dev.kumaflow_gold.transcript_embedding_index_demo"

## Prerequisite Steps for Setting Up a Delta Sync Index
To be able to set up a Delta Sync Index that points to our meeting transcript chunks, we first need to enable `Change Data Feed` on the gold table and add a primary key constraint on the `chunk_id` column.

In [0]:
# ── Prerequisites: Enable CDF and primary key on the source table ───────────────
# Delta Sync indexes require Change Data Feed and a declared primary key.

source_table = "dev.kumaflow_gold.transcript_chunks"

# Enable Change Data Feed (required for Delta Sync)
spark.sql(f"ALTER TABLE {source_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Add primary key constraint on chunk_id (required by Vector Search)
# The pk constraint requires a NOT NULL clause set on the column
spark.sql(f"ALTER TABLE {source_table} ALTER COLUMN chunk_id SET NOT NULL")
spark.sql(f"ALTER TABLE {source_table} ADD CONSTRAINT chunk_pk PRIMARY KEY (chunk_id)")

print(f"✅ Prerequisites complete for {source_table}")

In [0]:
# ── Create Vector Search endpoint ───────────────────────────────────────────────

# Create a Standard endpoint for low-latency semantic search
w.vector_search_endpoints.create_endpoint(
    name=endpoint_name,
    endpoint_type=EndpointType.STANDARD,
)

# Wait for endpoint to come online (should be fast)
endpoint_timeout = 120
endpoint_interval = 10

start = time.time()

while True:
    if time.time() - start > endpoint_timeout:
        raise TimeoutError(f"Endpoint '{endpoint_name}' did not come online within {endpoint_timeout}s")

    endpoint_status = w.vector_search_endpoints.get_endpoint(endpoint_name=endpoint_name).endpoint_status

    if endpoint_status is None:
        print("Endpoint status not yet available, retrying...")
        time.sleep(endpoint_interval)
        continue

    state = endpoint_status.state
    print(f"Endpoint state: {state}")

    if state == EndpointStatusState.ONLINE:
        print("✅ Endpoint is online")
        break
    elif state == EndpointStatusState.OFFLINE:
        msg = endpoint_status.message or "(no message)"
        raise RuntimeError(f"Endpoint '{endpoint_name}' is OFFLINE: {msg}")

    time.sleep(endpoint_interval)

In [0]:
# ── Create Delta Sync index with managed embeddings ─────────────────────────────

# Create the index (skip if already exists from a prior run)
try:
    w.vector_search_indexes.create_index(
        name=index_name,
        endpoint_name=endpoint_name,
        primary_key="chunk_id",
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=source_table,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name="chunk_text",
                    # databricks-gte-large-en and databricks-bge-large-en are the other options
                    embedding_model_endpoint_name="databricks-qwen3-embedding-0-6b",
                )
            ],
            pipeline_type=PipelineType.TRIGGERED,
        ),
    )
    print(f"✅ Index creation initiated: {index_name}")
except Exception as e:
    if "already exists" in str(e):
        print(f"ℹ️ Index already exists: {index_name}")
    else:
        raise

source_count = spark.table(source_table).count()

start = time.time()
index_timeout = 1200
index_interval = 60

while True:
    if time.time() - start > index_timeout:
        raise TimeoutError(f"Index '{index_name}' did not fully sync within {index_timeout}s")

    status = w.vector_search_indexes.get_index(index_name=index_name).status
    ready = status.ready if status is not None else False
    indexed = (status.indexed_row_count or 0) if status is not None else 0
    print(f"Ready: {ready} | indexed {indexed}/{source_count} rows")

    if ready and indexed == source_count:
        print("✅ Index is ready and fully synced")
        break
    time.sleep(index_interval)

## Check Semantic Search Against AI Search Endpoint
Now that we have the AI Search Endpoint with a Delta Sync Index configured, we can test out whether semantic search works by querying the endpoint and inspecting the returned chunks.

In [0]:
# ── Test query: verify the index returns relevant chunks ────────────────────────
columns = ["chunk_id", "call_id", "call_date", "chunk_index", "chunk_text"]

results = w.vector_search_indexes.query_index(
    index_name=index_name,
    columns=columns,
    query_text="What security concerns did customers raise?",
    num_results=5,
)

print(f"Query: 'What security concerns did customers raise?'")
print(f"Top {len(results.result.data_array)} results:\n")

for row in results.result.data_array:
    score = row[-1]  # similarity score is always the last element
    print(f"Score: {score:.4f} | call_date: {row[2]} | chunk_index: {row[3]}")
    print(f"  {row[4][:200]}...")
    print()